# Project A — 01 Shared Setup & Training

**Run once by the designated project owner.** Based on `Project_A_Shared_Colab_Bootstrap_v0_1.ipynb` and the validated Cells 1–9. This notebook prepares CIFAR datasets, fixes the five-class split, trains the Practical 3 rotation-prediction SSL teacher, and freezes/verifies its checkpoint on Google Drive.

**Execution:** In Colab, select a GPU runtime and choose **Runtime → Run all**. If the runtime disconnects, run all again; SSL training resumes from the last saved epoch. Once the frozen checkpoint exists, later runs skip SSL training and verify it instead. Other students should run only their student notebooks, which invoke the separate environment-validation logic.

**Caution:** The CIFAR data stored under `/content/aml_data` are temporary; Google Drive artifacts are persistent. Do not alter the frozen configuration/split/checkpoint after experiments start.

In [1]:
# Cell 1 — Mount Google Drive and locate the project
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PROJECT_ROOT = Path('/content/drive/MyDrive/Colab Notebooks/AI_Group_Project')
assert PROJECT_ROOT.is_dir(), f'Project directory not found: {PROJECT_ROOT}'
print('Project root:', PROJECT_ROOT)

Mounted at /content/drive
Project root: /content/drive/MyDrive/Colab Notebooks/AI_Group_Project


In [2]:
# Cell 2 — Dependencies, reproducibility and device
import sys, json, hashlib, random, datetime, os, re
import numpy as np
import torch, torchvision
import torch.nn.functional as F
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision.datasets import CIFAR10, CIFAR100
from torchvision.transforms import v2
SEED = 0
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Python:',sys.version.split()[0], 'Torch:',torch.__version__, 'Torchvision:',torchvision.__version__, 'Device:',DEVICE)
if DEVICE.type=='cuda': print('GPU:',torch.cuda.get_device_name(0))

Python: 3.13.16 Torch: 2.14.1+cu130 Torchvision: 0.29.1+cu130 Device: cuda
GPU: Tesla T4


In [3]:
# Cell 3 — Project directories and frozen-compatible configuration
SUBDIRS=['config','checkpoints/ssl','checkpoints/students','checkpoints/teachers','experiments/RQ1','experiments/RQ2','manifests','notebooks','reports','results/RQ1','results/RQ2','shared']
for subdir in SUBDIRS: (PROJECT_ROOT/subdir).mkdir(parents=True,exist_ok=True)
CONFIG_PATH=PROJECT_ROOT/'config/project_config.json'
proposed_cfg={
    'version':'draft-v0.1','seed':SEED,'source_dataset':'CIFAR10','target_dataset':'CIFAR100',
    'target_classes':['aquarium_fish','flatfish','ray','shark','trout'],
    'target_train_fraction':0.8,
    'ssl_pretext':'rotation_prediction_4way',
    'teacher_architecture':'ConvBackbone(num_features=64)',
    'preprocessing':'P3 rotation SSL: train random crop/flip/color/grayscale; normalized mean=(0.5,0.5,0.5), std=(0.2,0.2,0.2); validation deterministic',
    'checkpoint_format':'state_dict_plus_metadata'
}
if CONFIG_PATH.exists():
    cfg=json.loads(CONFIG_PATH.read_text(encoding='utf-8'))
    for key in ['seed','source_dataset','target_dataset','target_classes','target_train_fraction']:
        assert cfg[key]==proposed_cfg[key],f'Existing config mismatch: {key}'
    updates={key:value for key,value in proposed_cfg.items() if cfg.get(key) in (None,'PENDING_P3_REVIEW') and key in ('ssl_pretext','teacher_architecture','preprocessing')}
    if updates:
        assert not (PROJECT_ROOT/'checkpoints/ssl/teacher_ssl_v1.ckpt').exists(), 'Cannot modify config after teacher is frozen'
        cfg.update(updates)
        CONFIG_PATH.write_text(json.dumps(cfg,indent=2)+'\n',encoding='utf-8')
        print('Finalized pending P3 configuration metadata:',list(updates))
    for key in ('ssl_pretext','teacher_architecture'):
        assert cfg[key]==proposed_cfg[key],f'Unexpected SSL configuration: {key}'
else:
    cfg=proposed_cfg
    CONFIG_PATH.write_text(json.dumps(cfg,indent=2)+'\n',encoding='utf-8')
    print('Created project config')
print('Config:',CONFIG_PATH)

Finalized pending P3 configuration metadata: ['ssl_pretext', 'teacher_architecture', 'preprocessing']
Config: /content/drive/MyDrive/Colab Notebooks/AI_Group_Project/config/project_config.json


In [4]:
# Cell 4 — Download/load official datasets to local Colab storage
DATA_ROOT=Path('/content/aml_data')
source_train=CIFAR10(root=str(DATA_ROOT),train=True,download=True)
target_train=CIFAR100(root=str(DATA_ROOT),train=True,download=True)
target_test=CIFAR100(root=str(DATA_ROOT),train=False,download=True)
print('CIFAR10 source:',len(source_train),'CIFAR100 train:',len(target_train),'test:',len(target_test))

100%|██████████| 170M/170M [23:52<00:00, 119kB/s]
100%|██████████| 169M/169M [23:27<00:00, 120kB/s]


CIFAR10 source: 50000 CIFAR100 train: 50000 test: 10000


In [5]:
# Cell 5 — Preserve the original stratified CIFAR-100 target split
from sklearn.model_selection import train_test_split
classes=cfg['target_classes']
assert len(classes)==5 and len(set(classes))==5
assert all(c in target_train.classes for c in classes)
class_ids=[target_train.class_to_idx[c] for c in classes]
indices=np.flatnonzero(np.isin(np.asarray(target_train.targets),class_ids))
labels=np.asarray(target_train.targets)[indices]
train_idx,val_idx=train_test_split(
    indices,test_size=1-cfg['target_train_fraction'],
    random_state=cfg['seed'],stratify=labels)
test_idx=np.flatnonzero(np.isin(np.asarray(target_test.targets),class_ids))
manifest={
    'version':'draft-v0.1','seed':cfg['seed'],'classes':classes,
    'original_class_ids':class_ids,
    'train_indices':sorted(map(int,train_idx)),
    'val_indices':sorted(map(int,val_idx)),
    'test_indices':sorted(map(int,test_idx)),
    'label_map':{str(cid):i for i,cid in enumerate(class_ids)}
}
manifest_path=PROJECT_ROOT/'config/dataset_splits.json'
if manifest_path.exists():
    existing=json.loads(manifest_path.read_text(encoding='utf-8'))
    assert existing==manifest, 'Existing split manifest differs: STOP; do not overwrite'
    print('Existing validated dataset split manifest preserved')
else:
    manifest_path.write_text(json.dumps(manifest,indent=2),encoding='utf-8')
    print('Created dataset split manifest')
assert len(train_idx)==2000 and len(val_idx)==500 and len(test_idx)==500
assert not set(train_idx)&set(val_idx)
print('PASS — Dataset split validation')
print('Train / Val / Test:',len(train_idx),len(val_idx),len(test_idx))
print('Manifest:',manifest_path)

Existing validated dataset split manifest preserved
PASS — Dataset split validation
Train / Val / Test: 2000 500 500
Manifest: /content/drive/MyDrive/Colab Notebooks/AI_Group_Project/config/dataset_splits.json


In [6]:
# Cell 6 — Practical 3 SSL rotation-prediction teacher (architecture and training)
import copy
from PIL import Image
SSL_EPOCHS=30
SSL_BATCH_SIZE=128
SSL_LR=1e-3
SSL_WEIGHT_DECAY=1e-1
SSL_FEATURES=64
SSL_VAL_SIZE=5000
SSL_DIR=PROJECT_ROOT/'checkpoints/ssl'
SSL_DIR.mkdir(parents=True,exist_ok=True)
BEST_PATH=SSL_DIR/'ssl_training_best.ckpt'
LATEST_PATH=SSL_DIR/'ssl_training_latest.ckpt'
FROZEN_PATH=SSL_DIR/'teacher_ssl_v1.ckpt'

class ConvBackbone(nn.Module):
    def __init__(self,num_features=64,input_channels=3,dropout=0.3):
        super().__init__()
        self.conv1=nn.Conv2d(input_channels,32,3,padding=1)
        self.conv2=nn.Conv2d(32,64,3,padding=1)
        self.pool=nn.MaxPool2d(2,2)
        self.conv3=nn.Conv2d(64,128,3,padding=1)
        self.conv4=nn.Conv2d(128,256,3,padding=1)
        self.avg_pool=nn.AdaptiveAvgPool2d(1)
        self.fc1=nn.Linear(256,256)
        self.fc2=nn.Linear(256,128)
        self.dropout=nn.Dropout(dropout)
        self.features=nn.Linear(128,num_features)
    def forward(self,x):
        x=self.pool(F.relu(self.conv1(x)))
        x=self.pool(F.relu(self.conv2(x)))
        x=F.relu(self.conv3(x))
        x=F.relu(self.conv4(x))
        x=self.avg_pool(x)
        x=torch.flatten(x,1)
        x=F.relu(self.fc1(x))
        x=self.dropout(x)
        x=F.relu(self.fc2(x))
        x=self.dropout(x)
        return F.relu(self.features(x))

class ClassifierHead(nn.Module):
    def __init__(self,input_size=64,projection_size=32,num_classes=4):
        super().__init__()
        self.projection=nn.Linear(input_size,projection_size)
        self.classifier=nn.Linear(projection_size,num_classes)
    def forward(self,x):
        return self.classifier(F.relu(self.projection(x)))

def build_teacher_backbone():
    return ConvBackbone(num_features=SSL_FEATURES).to(DEVICE)

def build_rotation_model(backbone):
    return nn.Sequential(backbone,ClassifierHead(SSL_FEATURES,32,4)).to(DEVICE)

train_transform=v2.Compose([
    v2.ToImage(),v2.RandomResizedCrop(size=(32,32),scale=(0.8,1.0)),
    v2.RandomHorizontalFlip(p=0.5),
    v2.RandomApply([v2.ColorJitter(0.8,0.8,0.8,0.2)],p=0.5),
    v2.RandomGrayscale(p=0.2),v2.ToDtype(torch.float32,scale=True),
    v2.Normalize(mean=[0.5]*3,std=[0.2]*3)])
val_transform=v2.Compose([v2.ToImage(),v2.ToDtype(torch.float32,scale=True),v2.Normalize(mean=[0.5]*3,std=[0.2]*3)])

class RotationDataset(Dataset):
    def __init__(self,source,indices,training=True,seed=0):
        self.source=source
        self.indices=list(map(int,indices))
        self.training=training
        self.seed=seed
    def __len__(self):return len(self.indices)
    def __getitem__(self,index):
        image,_=self.source[self.indices[index]]
        rotation=int(np.random.randint(0,4)) if self.training else (index+self.seed)%4
        image=image.rotate(90*rotation)
        transform=train_transform if self.training else val_transform
        return transform(image),rotation

def build_ssl_loaders(source_dataset):
    rng=np.random.default_rng(SEED)
    indices=rng.permutation(len(source_dataset))
    train_data=RotationDataset(source_dataset,indices[SSL_VAL_SIZE:],True,SEED)
    val_data=RotationDataset(source_dataset,indices[:SSL_VAL_SIZE],False,SEED)
    train_loader=DataLoader(train_data,batch_size=SSL_BATCH_SIZE,shuffle=True,num_workers=2,pin_memory=(DEVICE.type=='cuda'))
    val_loader=DataLoader(val_data,batch_size=SSL_BATCH_SIZE,shuffle=False,num_workers=2,pin_memory=(DEVICE.type=='cuda'))
    return train_loader,val_loader

@torch.no_grad()
def evaluate_rotation(model,loader):
    model.eval()
    total_loss=total_correct=total_samples=0
    for images,labels in loader:
        images=images.to(DEVICE,non_blocking=True)
        labels=labels.to(DEVICE,non_blocking=True)
        logits=model(images)
        loss=F.cross_entropy(logits,labels)
        count=labels.size(0)
        total_loss+=loss.item()*count
        total_correct+=(logits.argmax(1)==labels).sum().item()
        total_samples+=count
    return total_loss/total_samples,total_correct/total_samples

def atomic_torch_save(payload,path):
    tmp=path.with_suffix('.tmp')
    torch.save(payload,tmp)
    os.replace(tmp,path)

def save_training_state(path,model,optimizer,epoch,best_accuracy,history,best_backbone_state):
    atomic_torch_save({
        'epoch':epoch,
        'model_state_dict':{k:v.detach().cpu().clone() for k,v in model.state_dict().items()},
        'optimizer_state_dict':optimizer.state_dict(),
        'best_val_accuracy':best_accuracy,'history':history,'seed':SEED,
        'ssl_pretext':'rotation_prediction_4way','epochs_target':SSL_EPOCHS,
        'best_backbone_state':best_backbone_state,
        'torch_rng_state':torch.get_rng_state(),
        'numpy_rng_state':np.random.get_state(),
        'python_rng_state':random.getstate(),
        'cuda_rng_state':torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
    },path)

def train_ssl_teacher(backbone,source_dataset,device=DEVICE,epochs=SSL_EPOCHS):
    assert torch.cuda.is_available() and torch.device(device).type=='cuda','CUDA GPU required for this run'
    if FROZEN_PATH.exists():
        raise FileExistsError('Frozen SSL teacher exists; use Cell 10 skip-and-verify path')
    train_loader,val_loader=build_ssl_loaders(source_dataset)
    model=build_rotation_model(backbone).to(device)
    optimizer=torch.optim.AdamW(model.parameters(),lr=SSL_LR,weight_decay=SSL_WEIGHT_DECAY)
    history=[];best_accuracy=-1.0;best_backbone_state=None;start_epoch=1
    if LATEST_PATH.exists():
        saved=torch.load(LATEST_PATH,map_location='cpu',weights_only=False)  # locally generated trusted project checkpoint; contains RNG tuples
        assert saved['seed']==SEED and saved['ssl_pretext']=='rotation_prediction_4way'
        assert saved.get('epochs_target',epochs)==epochs,'Epoch target mismatch'
        model.load_state_dict(saved['model_state_dict'],strict=True)
        optimizer.load_state_dict(saved['optimizer_state_dict'])
        # Optimizer tensors must reside on the parameter device after restoring.
        for state in optimizer.state.values():
            for k,v in state.items():
                if torch.is_tensor(v): state[k]=v.to(device)
        history=saved['history'];best_accuracy=saved['best_val_accuracy']
        best_backbone_state=saved.get('best_backbone_state')
        if best_backbone_state is None and BEST_PATH.exists():
            best_saved=torch.load(BEST_PATH,map_location='cpu',weights_only=False)
            best_backbone_state={k[2:]:v for k,v in best_saved['model_state_dict'].items() if k.startswith('0.')}
        start_epoch=int(saved['epoch'])+1
        if 'torch_rng_state' in saved: torch.set_rng_state(saved['torch_rng_state'])
        if 'numpy_rng_state' in saved: np.random.set_state(saved['numpy_rng_state'])
        if 'python_rng_state' in saved: random.setstate(saved['python_rng_state'])
        if torch.cuda.is_available() and saved.get('cuda_rng_state') is not None: torch.cuda.set_rng_state_all(saved['cuda_rng_state'])
        print(f'Resuming SSL from completed epoch {start_epoch-1}/{epochs}')
    else:
        torch.manual_seed(SEED);np.random.seed(SEED);random.seed(SEED)
        if torch.cuda.is_available():torch.cuda.manual_seed_all(SEED)
        print('Starting new SSL training run')
    print('SSL training:',len(train_loader.dataset),'images; validation:',len(val_loader.dataset),'images')
    for epoch in range(start_epoch,epochs+1):
        model.train();train_loss_sum=train_correct=train_samples=0
        for images,labels in train_loader:
            images=images.to(device,non_blocking=True);labels=labels.to(device,non_blocking=True)
            optimizer.zero_grad();logits=model(images);loss=F.cross_entropy(logits,labels)
            loss.backward();optimizer.step()
            count=labels.size(0)
            train_loss_sum+=loss.item()*count
            train_correct+=(logits.argmax(1)==labels).sum().item()
            train_samples+=count
        train_loss=train_loss_sum/train_samples
        train_acc=train_correct/train_samples
        val_loss,val_acc=evaluate_rotation(model,val_loader)
        history.append({'epoch':epoch,'train_loss':train_loss,'train_accuracy':train_acc,'val_loss':val_loss,'val_accuracy':val_acc})
        if val_acc>best_accuracy:
            best_accuracy=val_acc
            best_backbone_state={k:v.detach().cpu().clone() for k,v in model[0].state_dict().items()}
            save_training_state(BEST_PATH,model,optimizer,epoch,best_accuracy,history,best_backbone_state)
        save_training_state(LATEST_PATH,model,optimizer,epoch,best_accuracy,history,best_backbone_state)
        print(f'Epoch {epoch:02d}/{epochs} | Train loss {train_loss:.4f} | Train acc {train_acc:.2%} | Val loss {val_loss:.4f} | Val acc {val_acc:.2%} | Best {best_accuracy:.2%}',flush=True)
    assert len(history)==epochs and best_backbone_state is not None,'Incomplete SSL training or missing best weights'
    backbone.load_state_dict(best_backbone_state,strict=True)
    backbone.eval()
    info={'pretext':'rotation_prediction_4way','epochs':epochs,'seed':SEED,'best_val_accuracy':best_accuracy,
          'train_images':len(train_loader.dataset),'val_images':len(val_loader.dataset),
          'batch_size':SSL_BATCH_SIZE,'learning_rate':SSL_LR,'weight_decay':SSL_WEIGHT_DECAY,'history':history}
    print('SSL training complete. Best validation accuracy:',f'{best_accuracy:.2%}')
    return backbone,info

teacher=build_teacher_backbone()
parameter_count=sum(p.numel() for p in teacher.parameters())
with torch.no_grad():
    teacher.eval()
    dummy=torch.zeros(2,3,32,32,device=DEVICE)
    features=teacher(dummy)
    logits=build_rotation_model(teacher)(dummy)
assert parameter_count==495360
assert features.shape==(2,64) and logits.shape==(2,4)
print('PASS — SSL architecture validation')
print('Backbone parameters:',parameter_count,'Backbone output:',tuple(features.shape),'Rotation logits:',tuple(logits.shape))

PASS — SSL architecture validation
Backbone parameters: 495360 Backbone output: (2, 64) Rotation logits: (2, 4)


In [7]:
# Cell 7 — Frozen teacher checkpoint utilities

def sha256_file(path):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
    return h.hexdigest()

def save_ssl_checkpoint(backbone,training_info,path=None):
    path=Path(path or FROZEN_PATH)
    path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():raise FileExistsError(f'Frozen checkpoint already exists: {path}')
    payload={
        'model_state_dict':{k:v.detach().cpu().clone() for k,v in backbone.state_dict().items()},
        'config':cfg,'training_info':training_info,'torch_version':torch.__version__,
        'created_utc':datetime.datetime.now(datetime.timezone.utc).isoformat()}
    atomic_torch_save(payload,path)
    digest=sha256_file(path)
    path.with_suffix('.sha256').write_text(digest+'  '+path.name+'\n',encoding='utf-8')
    print('Saved:',path,'SHA256:',digest)
    return path
print('PASS — Cell 7 checkpoint utilities initialized.')

PASS — Cell 7 checkpoint utilities initialized.


In [11]:

# Cell 8 — Verify and load the frozen SSL teacher

from torch.torch_version import TorchVersion

CKPT = FROZEN_PATH

def load_verified_ssl_teacher(path=CKPT):
    path = Path(path)
    checksum_path = path.with_suffix('.sha256')

    assert path.is_file(), f'Checkpoint missing: {path}'
    assert checksum_path.is_file(), f'SHA256 file missing: {checksum_path}'

    actual = sha256_file(path)
    expected = checksum_path.read_text(encoding='utf-8').split()[0]
    assert actual == expected, 'Checkpoint SHA256 mismatch'

    # Allowlist PyTorch's TorchVersion for the trusted checkpoint.
    # Keep weights_only=True for restricted deserialization.
    with torch.serialization.safe_globals([TorchVersion]):
        payload = torch.load(
            path,
            map_location='cpu',
            weights_only=True
        )

    assert payload['config']['target_classes'] == cfg['target_classes'], \
        'Target-class mismatch'
    assert payload['config']['seed'] == SEED, \
        'Seed mismatch'

    teacher = build_teacher_backbone()
    teacher.load_state_dict(
        payload['model_state_dict'],
        strict=True
    )
    teacher.eval()

    with torch.no_grad():
        features = teacher(
            torch.zeros(
                2, 3, 32, 32,
                device=next(teacher.parameters()).device
            )
        )

    assert features.shape == (2, 64), \
        f'Unexpected output: {features.shape}'

    print('PASS — Frozen SSL checkpoint verified')
    print('SHA256:', actual, 'Backbone output:', tuple(features.shape))
    print(
        'Best SSL validation accuracy:',
        payload['training_info'].get(
            'best_val_accuracy', 'Not recorded'
        )
    )

    return teacher, payload


if CKPT.exists():
    teacher, ssl_payload = load_verified_ssl_teacher()
else:
    print('PASS — Cell 8 loader initialized; checkpoint pending training.')


PASS — Frozen SSL checkpoint verified
SHA256: 81570749345c9a47e35918106f6a273f244dc1b57918b59c7069239a855bbb36 Backbone output: (2, 64)
Best SSL validation accuracy: 0.5986


In [9]:
# Cell 9 — Reproducibility environment manifest
MEMBER='student1'  # Designated setup owner; adjust if needed
assert re.fullmatch(r'[A-Za-z0-9_-]+',MEMBER)
report={
    'member':MEMBER,
    'timestamp_utc':datetime.datetime.now(datetime.timezone.utc).isoformat(),
    'python':sys.version.split()[0], 'torch':torch.__version__,
    'torchvision':torchvision.__version__,
    'cuda_available':torch.cuda.is_available(),
    'gpu_name':torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    'config_sha256':sha256_file(CONFIG_PATH),
    'split_sha256':sha256_file(manifest_path),
    'ssl_checkpoint_sha256':sha256_file(CKPT) if CKPT.exists() else None,
    'ssl_checkpoint_status':'available' if CKPT.exists() else 'pending_training'
}
manifest_dir=PROJECT_ROOT/'manifests';manifest_dir.mkdir(parents=True,exist_ok=True)
report_path=manifest_dir/f'{MEMBER}_environment.json'
report_path.write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
print(json.dumps(report,indent=2))
print('PASS — Environment report generated:',report_path)

{
  "member": "student1",
  "timestamp_utc": "2026-11-20T13:07:31.625738+00:00",
  "python": "3.13.16",
  "torch": "2.14.1+cu130",
  "torchvision": "0.29.1+cu130",
  "cuda_available": true,
  "gpu_name": "Tesla T4",
  "config_sha256": "25f4d3c7556b80b8c5b4ea9423fbf38ab28cf6ff7e93ddfa0dc6f8a9c7ed24bb",
  "split_sha256": "8967db46b9cd7a98e88bcada7e5a6f974c312110a116aca6cabb66b59d5639ef",
  "ssl_checkpoint_sha256": null,
  "ssl_checkpoint_status": "pending_training"
}
PASS — Environment report generated: /content/drive/MyDrive/Colab Notebooks/AI_Group_Project/manifests/student1_environment.json


## Execute the one-time SSL training

Cell 10 automatically skips training if a valid frozen checkpoint exists. Otherwise it resumes from `ssl_training_latest.ckpt` when available, or starts a new 30-epoch training run. It then freezes the best-validation backbone and verifies its checksum. **Run all will execute this training**; ensure the Colab GPU runtime is enabled first.

The resume checkpoint captures optimizer/model and RNG states at epoch boundaries. Because DataLoader workers and augmentation RNGs are not restored exactly, resumed runs are reproducible at the artifact level but not guaranteed bit-for-bit identical to uninterrupted training.

In [12]:
# Cell 10 — One-time SSL training, automatic resume and frozen checkpoint creation
if CKPT.exists():
    print('Frozen teacher already exists — skipping SSL training.')
    teacher,ssl_payload=load_verified_ssl_teacher()
    training_info=ssl_payload['training_info']
else:
    assert DEVICE.type=='cuda','Select Runtime → Change runtime type → GPU before training'
    backbone=build_teacher_backbone()
    backbone,training_info=train_ssl_teacher(backbone,source_train,device=DEVICE,epochs=SSL_EPOCHS)
    CKPT=save_ssl_checkpoint(backbone,training_info)
    teacher,ssl_payload=load_verified_ssl_teacher(CKPT)
print('PASS — SSL teacher available for downstream experiments')

Frozen teacher already exists — skipping SSL training.
PASS — Frozen SSL checkpoint verified
SHA256: 81570749345c9a47e35918106f6a273f244dc1b57918b59c7069239a855bbb36 Backbone output: (2, 64)
Best SSL validation accuracy: 0.5986
PASS — SSL teacher available for downstream experiments


In [13]:
# Cell 11 — Finalize environment manifest with the frozen checkpoint hash
assert CKPT.exists() and CKPT.with_suffix('.sha256').exists()
report['ssl_checkpoint_sha256']=sha256_file(CKPT)
report['ssl_checkpoint_status']='available'
report['timestamp_utc']=datetime.datetime.now(datetime.timezone.utc).isoformat()
report_path.write_text(json.dumps(report,indent=2)+'\n',encoding='utf-8')
print('PASS — Final environment manifest updated:',report_path)
print('Frozen checkpoint SHA256:',report['ssl_checkpoint_sha256'])

PASS — Final environment manifest updated: /content/drive/MyDrive/Colab Notebooks/AI_Group_Project/manifests/student1_environment.json
Frozen checkpoint SHA256: 81570749345c9a47e35918106f6a273f244dc1b57918b59c7069239a855bbb36


In [14]:
# Cell 12 — Final shared-artifact inventory
required=[CONFIG_PATH,manifest_path,CKPT,CKPT.with_suffix('.sha256'),report_path]
for path in required:
    assert path.is_file(),f'Missing shared artifact: {path}'
    print('PASS',path.relative_to(PROJECT_ROOT),f'({path.stat().st_size:,} bytes)')
print('\nPASS — Notebook 1 completed: shared artifacts prepared and frozen SSL teacher verified.')

PASS config/project_config.json (547 bytes)
PASS config/dataset_splits.json (32,259 bytes)
PASS checkpoints/ssl/teacher_ssl_v1.ckpt (1,989,253 bytes)
PASS checkpoints/ssl/teacher_ssl_v1.sha256 (86 bytes)
PASS manifests/student1_environment.json (525 bytes)

PASS — Notebook 1 completed: shared artifacts prepared and frozen SSL teacher verified.
